# 13 — LayoutLMv3 Field Extraction Playground

Load the trained LayoutLMv3 model, inspect raw predictions, and apply
`InvoiceCleaner` — the state-of-the-art post-processing module.

| | |
|---|---|
| **Model** | Best checkpoint from Notebook 12 |
| **Cleaner** | `src/invoice_cleaner.py` — `InvoiceCleaner` class |
| **No retraining** | Weights loaded from disk, read-only |
| **No generative AI** | Discriminative token classifier only |

**Prerequisite:** copy `invoice_cleaner.py` into your `src/` folder.

**Workflow:**
1. Run Cells 1-4 once after kernel restart (loads model + cleaner)
2. Cell 5 shows raw model output — no cleaning applied
3. Cell 6 shows RAW vs CLEANED side by side using `InvoiceCleaner`
4. Cell 7 tests on your own images from disk
5. Cell 8 shows token-level debug when something looks wrong


## 1. Environment — run this first after every kernel restart

In [1]:
import os, time

# Must be set before any transformers/tokenizers import
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

t0 = time.time()
from transformers import LayoutLMv3ForTokenClassification, LayoutLMv3Processor
print('transformers imported in', round(time.time() - t0, 2), 'sec')
print('Python:', __import__('sys').executable)


/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/.venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers imported in 3.34 sec
Python: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/.venv311/bin/python


## 2. Paths, labels and dataset

In [2]:
import json
import sys
import torch
from pathlib import Path
from PIL import Image
from datasets import load_from_disk

PROJECT_ROOT = Path('..').resolve()
DATASET_DIR  = PROJECT_ROOT / 'data' / 'processed' / 'layoutlmv3_dataset'
CKPT_PATH    = PROJECT_ROOT / 'models' / 'experimental' / 'layoutlmv3_fatura' / 'best_checkpoint'
MAX_LENGTH   = 512

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else 'cpu'
)

with open(DATASET_DIR / 'label2id.json') as f:
    label2id = json.load(f)
with open(DATASET_DIR / 'id2label.json') as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

BIO_LABELS = [id2label[i] for i in range(len(id2label))]

FIELD_ORDER = [
    'INVOICE_NUMBER', 'INVOICE_DATE', 'DUE_DATE',
    'ISSUER_NAME',    'RECIPIENT_NAME', 'TOTAL_AMOUNT'
]

# Mapping from model uppercase keys to InvoiceCleaner lowercase keys
CLEAN_KEY = {
    'INVOICE_NUMBER': 'invoice_number',
    'INVOICE_DATE':   'invoice_date',
    'DUE_DATE':       'due_date',
    'ISSUER_NAME':    'issuer_name',
    'RECIPIENT_NAME': 'recipient_name',
    'TOTAL_AMOUNT':   'total_amount',
}

raw_dataset = load_from_disk(str(DATASET_DIR))

# ── Import InvoiceCleaner from src/ ───────────────────────────────────
# invoice_cleaner.py must be placed in PROJECT_ROOT/src/
SRC_DIR = str(PROJECT_ROOT / 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from invoice_cleaner import InvoiceCleaner
cleaner = InvoiceCleaner()

print(f'Device     : {DEVICE}')
print(f'Checkpoint : {CKPT_PATH}')
print(f'Exists     : {CKPT_PATH.exists()}')
print(f'Labels     : {BIO_LABELS}')
print(f'Dataset    : {raw_dataset}')
print(f'Cleaner    : {cleaner.__class__.__name__} ready')


Device     : mps
Checkpoint : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/layoutlmv3_fatura/best_checkpoint
Exists     : True
Labels     : ['O', 'B-INVOICE_NUMBER', 'I-INVOICE_NUMBER', 'B-INVOICE_DATE', 'I-INVOICE_DATE', 'B-DUE_DATE', 'I-DUE_DATE', 'B-ISSUER_NAME', 'I-ISSUER_NAME', 'B-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'B-TOTAL_AMOUNT', 'I-TOTAL_AMOUNT']
Dataset    : DatasetDict({
    train: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 1734
    })
    val: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 371
    })
    test: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 372
    })
})
Cleaner    : InvoiceCleaner ready


## 3. Load model and processor from checkpoint

In [3]:
# Loads from disk only — no network, no retraining needed.

processor = LayoutLMv3Processor.from_pretrained(
    str(CKPT_PATH),
    apply_ocr=False,
    use_fast=True,
    local_files_only=True,
)

model = LayoutLMv3ForTokenClassification.from_pretrained(
    str(CKPT_PATH),
    id2label=id2label,
    label2id=label2id,
    local_files_only=True,
)
model.to(DEVICE)
model.eval()

print('Model loaded OK')
print('  tokenizer :', type(processor.tokenizer).__name__)
print('  num labels:', model.config.num_labels)
print('  device    :', DEVICE)


The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
Loading weights: 100%|██████████| 216/216 [00:00<00:00, 7872.52it/s]


Model loaded OK
  tokenizer : LayoutLMv3Tokenizer
  num labels: 13
  device    : mps


## 4. Core inference functions — do not edit this cell

In [4]:
def get_raw_predictions(image, words, bboxes):
    """
    Run LayoutLMv3 on image + OCR words/boxes.
    Returns dict {FIELD_NAME: raw_string} with NO cleaning applied.
    This is the pure model output — the ground truth of what the model learned.
    """
    encoding = processor(
        image, words, boxes=bboxes,
        truncation=True, padding='max_length',
        max_length=MAX_LENGTH, return_tensors='pt',
    )
    with torch.no_grad():
        outputs = model(**{k: v.to(DEVICE) for k, v in encoding.items()})

    token_preds = outputs.logits.argmax(-1).squeeze(0).cpu().tolist()
    word_ids    = encoding.word_ids(batch_index=0)

    # Map subword tokens back to word level (take first subword per word)
    word_preds = {}
    for ti, wi in enumerate(word_ids):
        if wi is not None and wi not in word_preds:
            word_preds[wi] = token_preds[ti]

    aligned_words    = [words[i]      for i in sorted(word_preds)]
    aligned_pred_ids = [word_preds[i] for i in sorted(word_preds)]

    # Group BIO tokens into field strings
    fields, current_field, current_tokens = {}, None, []
    for label_id, word in zip(aligned_pred_ids, aligned_words):
        label = id2label[label_id]
        if label == 'O':
            if current_field:
                text = ' '.join(current_tokens).strip()
                if text:
                    fields[current_field] = text
                current_field, current_tokens = None, []
        elif label.startswith('B-'):
            if current_field:
                text = ' '.join(current_tokens).strip()
                if text:
                    fields[current_field] = text
            current_field, current_tokens = label[2:], [word]
        elif label.startswith('I-'):
            fn = label[2:]
            if current_field == fn:
                current_tokens.append(word)
            elif current_field is None and fn in fields:
                fields[fn] += ' ' + word
            elif current_field is None:
                current_field, current_tokens = fn, [word]
            else:
                text = ' '.join(current_tokens).strip()
                if text:
                    fields[current_field] = text
                current_field, current_tokens = fn, [word]
    if current_field:
        text = ' '.join(current_tokens).strip()
        if text:
            fields[current_field] = text

    return fields


def token_debug(image, words, bboxes):
    """Show every non-O token prediction — useful for debugging."""
    encoding = processor(
        image, words, boxes=bboxes,
        truncation=True, padding='max_length',
        max_length=MAX_LENGTH, return_tensors='pt',
    )
    with torch.no_grad():
        outputs = model(**{k: v.to(DEVICE) for k, v in encoding.items()})
    token_preds = outputs.logits.argmax(-1).squeeze(0).cpu().tolist()
    word_ids    = encoding.word_ids(batch_index=0)
    word_preds  = {}
    for ti, wi in enumerate(word_ids):
        if wi is not None and wi not in word_preds:
            word_preds[wi] = token_preds[ti]

    print(f"{'LABEL':<28}  WORD")
    print('-' * 55)
    for wi in sorted(word_preds):
        label = id2label[word_preds[wi]]
        if label != 'O':
            print(f"  {label:<28}: {repr(words[wi])}")


print('Core inference functions defined OK')


Core inference functions defined OK


## 5. Raw model output on test set examples

No cleaning applied — this is exactly what the model produces.
Change `N_EXAMPLES` or `SPLIT` as needed.


In [5]:
N_EXAMPLES = 5      # how many examples to show
SPLIT      = 'test' # 'train', 'val', or 'test'

for i in range(N_EXAMPLES):
    example = raw_dataset[SPLIT][i]
    image   = Image.open(example['image_path']).convert('RGB')
    words   = example['words']
    bboxes  = example['bboxes']

    raw = get_raw_predictions(image, words, bboxes)

    print(f"\n{'='*60}")
    print(f"  {SPLIT.upper()} {i}: {Path(example['image_path']).stem}")
    print('='*60)
    for field in FIELD_ORDER:
        value = raw.get(field, '—')
        print(f"  {field:<20}: {value}")



  TEST 0: Template1_Instance189
  INVOICE_NUMBER      : —
  INVOICE_DATE        : Date 29-Apr-2012
  DUE_DATE            : Due Date 07-Aug-2010
  ISSUER_NAME         : —
  RECIPIENT_NAME      : Bill to Shelly Rodriguez 02547 Ramos Bypass Suite 849 Williamshaven, NC 38767 US Tel +(463)893-0347 Email christina14@example.org Site http //www.gomez.com/
  TOTAL_AMOUNT        : TOTAL 134.41 USD

  TEST 1: Template38_Instance29
  INVOICE_NUMBER      : INVOICE # 9Y1M9d-232
  INVOICE_DATE        : Date 15-Feb-1993
  DUE_DATE            : Due Date 14-Jan-2007
  ISSUER_NAME         : —
  RECIPIENT_NAME      : —
  TOTAL_AMOUNT        : BALANCE DUE 220.90 EUR

  TEST 2: Template28_Instance30
  INVOICE_NUMBER      : —
  INVOICE_DATE        : Invoice Date 13-Aug-2002
  DUE_DATE            : —
  ISSUER_NAME         : —
  RECIPIENT_NAME      : Bill to Daniel Moore 04274 Claudia Fort Suite 045 Patriciashire, SD 32054 US Tel +(491)040-2728 Email frichardson@example.org Site https //berger-bailey.com/
  

## 6. InvoiceCleaner output — RAW vs CLEANED side by side

Uses `InvoiceCleaner` from `src/invoice_cleaner.py`.
No hand-rolled regex here — the cleaner handles everything.

**What the cleaner does per field:**
- `INVOICE_DATE` / `DUE_DATE` — extracts date pattern from raw string (5 formats)
- `TOTAL_AMOUNT` — extracts number + currency (8 format variants)
- `INVOICE_NUMBER` — takes the last alphanumeric token
- `ISSUER_NAME` / `RECIPIENT_NAME` — skips label words, stops at address tokens
- **OCR fallbacks:** recipient from bill-to keywords, missing date from word stream,
  swapped-date arbitration when model tags both dates as the same field

Change `N_EXAMPLES` or `SPLIT` and re-run freely.


In [6]:
N_EXAMPLES = 5
SPLIT      = 'test'

for i in range(N_EXAMPLES):
    example = raw_dataset[SPLIT][i]
    image   = Image.open(example['image_path']).convert('RGB')
    words   = example['words']
    bboxes  = example['bboxes']

    raw     = get_raw_predictions(image, words, bboxes)

    # Pass ocr_words so fallback extractors can run:
    # - recipient fallback (model missed RECIPIENT_NAME)
    # - date fallback (model only captured label, not value)
    # - swapped-date arbitration (both dates tagged as same field)
    cleaned = cleaner.clean(raw, ocr_words=words)

    print(f"\n{'='*75}")
    print(f"  {SPLIT.upper()} {i}: {Path(example['image_path']).stem}")
    print('='*75)
    print(f"  {'FIELD':<20}  {'RAW MODEL OUTPUT':<32}  CLEANED")
    print(f"  {'-'*20}  {'-'*32}  {'-'*30}")
    for field in FIELD_ORDER:
        raw_val     = raw.get(field, '—')
        cleaned_val = cleaned.get(CLEAN_KEY[field], '') or '—'
        raw_display = (raw_val[:30] + '..') if len(raw_val) > 32 else raw_val
        print(f"  {field:<20}  {raw_display:<32}  {cleaned_val}")



  TEST 0: Template1_Instance189
  FIELD                 RAW MODEL OUTPUT                  CLEANED
  --------------------  --------------------------------  ------------------------------
  INVOICE_NUMBER        —                                 —
  INVOICE_DATE          Date 29-Apr-2012                  29-Apr-2012
  DUE_DATE              Due Date 07-Aug-2010              07-Aug-2010
  ISSUER_NAME           —                                 —
  RECIPIENT_NAME        Bill to Shelly Rodriguez 02547..  Shelly Rodriguez
  TOTAL_AMOUNT          TOTAL 134.41 USD                  134.41 USD

  TEST 1: Template38_Instance29
  FIELD                 RAW MODEL OUTPUT                  CLEANED
  --------------------  --------------------------------  ------------------------------
  INVOICE_NUMBER        INVOICE # 9Y1M9d-232              9Y1M9d-232
  INVOICE_DATE          Date 15-Feb-1993                  15-Feb-1993
  DUE_DATE              Due Date 14-Jan-2007              14-Jan-2007
  ISSUER_NA

## 7. Test on your own images from disk

Put your image paths in `MY_IMAGES` and run.
Pipeline: Tesseract OCR → `get_raw_predictions()` → `InvoiceCleaner`.

Requires Cells 1-4 and Cell 4 (inference functions) to have been run.


In [14]:
import pytesseract
import subprocess
import sys
import re

try:
    from pdf2image import convert_from_path
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pdf2image', '-q'])
    from pdf2image import convert_from_path

try:
    import fitz
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pymupdf', '-q'])
    import fitz

MY_IMAGES = [
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/doc_i_2.avif',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/doc_i_1.webp',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/doc_i_3.webp',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/invoice-0-4.pdf',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/invoice-1-3.pdf',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/invoice-2-1.pdf',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/invoice-3-0.pdf',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/invoice-7-0.pdf',
]

_PDF_NATIVE_TEXT_THRESHOLD = 100

_DATE = (
    r'\d{1,2}\.\d{1,2}\.\d{4}'
    r'|\d{4}-\d{2}-\d{2}'
    r'|\d{1,2}/\d{1,2}/\d{2,4}'
    r'|(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|'
    r'Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|'
    r'Nov(?:ember)?|Dec(?:ember)?)\s+\d{1,2},?\s+\d{4}'
    r'|\d{1,2}\s+(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|'
    r'Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|'
    r'Nov(?:ember)?|Dec(?:ember)?)\s+\d{4}'
)
_DATE_RE = re.compile(_DATE, re.IGNORECASE)


def _is_native_pdf(path):
    doc = fitz.open(str(path))
    total_chars = sum(len(page.get_text()) for page in doc)
    doc.close()
    return total_chars >= _PDF_NATIVE_TEXT_THRESHOLD


def _extract_fields_from_text(text):
    """
    Regex extraction from native PDF text.
    Handles 5 date formats, 3-column table layouts, labelled and unlabelled fields.
    """
    fields = {}
    lines = [l.strip() for l in text.split('\n')]

    # ── Invoice number ─────────────────────────────────────────────────
    for pat in [
        r'invoice\s*(?:no\.?|number|#|num\.?)\s*:?\s+([A-Z0-9#][A-Z0-9#\-/]+)',
        r'INVOICE\s+#\s+([A-Z0-9][A-Z0-9\-]+)',
        r'^INVOICE\s+(\d{4,})',
        r'^invoice\s+(?:no\.?\s+)?(\d{3,})',
    ]:
        m = re.search(pat, text, re.IGNORECASE | re.MULTILINE)
        if m:
            val = m.group(1).strip()
            if any(c.isdigit() for c in val):
                fields['invoice_number'] = val
                break

    # ── Invoice date ───────────────────────────────────────────────────
    for pat in [
        r'(?:invoice\s+)?date\s*:?\s*\n?\s*(' + _DATE + r')',
        r'(?:issued?|created)\s*:?\s*(' + _DATE + r')',
    ]:
        m = re.search(pat, text, re.IGNORECASE | re.MULTILINE)
        if m:
            fields['invoice_date'] = m.group(1).strip()
            break

    if 'invoice_date' not in fields:
        m = re.search(r'^(' + _DATE + r')\s*\n\s*invoice', text, re.IGNORECASE | re.MULTILINE)
        if m:
            fields['invoice_date'] = m.group(1).strip()

    # 3-column table: Date / To / Ship To headers then values on next lines
    if 'invoice_date' not in fields:
        for i, line in enumerate(lines):
            if line.lower() == 'date' and i + 3 < len(lines):
                if lines[i+1].lower() in ('to', 'ship to') or lines[i+2].lower() in ('to', 'ship to'):
                    for j in range(i+2, min(i+5, len(lines))):
                        if _DATE_RE.match(lines[j]):
                            fields['invoice_date'] = lines[j].strip()
                            if j+1 < len(lines):
                                candidate = lines[j+1].strip()
                                if (candidate
                                    and 'same as' not in candidate.lower()
                                    and candidate.lower() not in ('to','ship to','none')
                                    and not re.match(r'^\d+\s+[A-Z]', candidate)
                                    and len(candidate) < 60):
                                    fields['recipient_name'] = candidate
                            break
                break

    # ── Due date ───────────────────────────────────────────────────────
    for pat in [
        r'due[\s\-]date\s*:?\s*(' + _DATE + r')',
        r'payment\s+due\s*:?\s*(' + _DATE + r')',
        r'total\s+due\s+by\s*[^\n]*?(' + _DATE + r')',
        r'pay(?:able)?\s+(?:by|on)\s*:?\s*(' + _DATE + r')',
    ]:
        m = re.search(pat, text, re.IGNORECASE | re.MULTILINE)
        if m:
            val = m.group(1).strip()
            if val != fields.get('invoice_date', ''):
                fields['due_date'] = val
                break

    if 'due_date' not in fields:
        m = re.search(
            r'order\s+date\s*\n+order\s+number\s*\n+due\s+date\s*\n+'
            r'(' + _DATE + r')\s*\n+\S+\s*\n+(' + _DATE + r')',
            text, re.IGNORECASE | re.MULTILINE
        )
        if m:
            if 'invoice_date' not in fields:
                fields['invoice_date'] = m.group(1).strip()
            fields['due_date'] = m.group(2).strip()

    # ── Issuer name ────────────────────────────────────────────────────
    SKIP = re.compile(
        r'^(?:tel|fax|phone|email|web|http|www|invoice|date|lorem|we\s|'
        r'order|ship|bill|to\b|from\b|due|total|sub)',
        re.IGNORECASE
    )
    PHONE = re.compile(r'^\+?\d[\d\s\-().]{5,}$')
    for line in lines[:20]:
        if not line or SKIP.match(line): continue
        if PHONE.match(line): continue
        if '@' in line or line.lower().startswith('http'): continue
        if re.match(r'^\d+\s+[A-Za-z]', line): continue
        if re.match(r'^\d', line): continue
        if len(line) > 60: continue
        fields['issuer_name'] = line
        break

    # ── Recipient name ─────────────────────────────────────────────────
    if 'recipient_name' not in fields:
        for pat in [
            r'bill\s+to\s*:?\s*\n\s*(.+)',
            r'billed\s+to\s*:?\s*\n\s*(.+)',
            r'ship\s+to\s*:?\s*\n\s*([A-Z][^\n]{1,50})',
            r'^to\s*\n\s*([A-Z][^\n]{1,50})',
        ]:
            m = re.search(pat, text, re.IGNORECASE | re.MULTILINE)
            if m:
                candidate = m.group(1).strip()
                if (candidate
                    and not re.match(r'^\d+\s+[A-Z]', candidate)
                    and '@' not in candidate
                    and 'same as' not in candidate.lower()
                    and candidate.lower() not in ('ship to','bill to','to','none')
                    and len(candidate) < 60):
                    fields['recipient_name'] = candidate
                    break

    # ── Total amount ───────────────────────────────────────────────────
    _CUR = r'(?:\s*(?:EUR|USD|GBP|CAD|AUD|CHF))?'
    for pat in [
        r'total\s+due\s+by[^\n]*\n\s*(\d[\d,\.]+)' + _CUR,
        r'total\s+due\s*:?\s*\n?\s*(\d[\d,\.]+)' + _CUR,
        r'amount\s+due\s*:?\s*\n?\s*(\d[\d,\.]+)' + _CUR,
        r'grand\s+total\s*:?\s*\$?\s*(\d[\d,\.]+)' + _CUR,
        r'\btotal\b\s*\**\s*\$?\s*(\d[\d,\.]+)' + _CUR,
        r'balance\s+due\s*:?\s*(\d[\d,\.]+)' + _CUR,
        r'total\s+due\s*\n\s*(\d[\d,\.]+)' + _CUR,
    ]:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            amount = m.group(1).strip().rstrip('.,')
            try:
                v = float(amount.replace(',', ''))
                if v < 1: continue
            except ValueError:
                continue
            # Reject small decimals that are date fragments (e.g. 24.09)
            if re.match(r'^\d{1,2}\.\d{2}$', amount) and float(amount) < 50:
                continue
            context = text[max(0, m.start()-20):m.end()+20]
            cur_m = re.search(r'\b(EUR|USD|GBP|CAD|AUD)\b|[$€£]', context, re.IGNORECASE)
            currency = ''
            if cur_m:
                c = cur_m.group(0).upper()
                currency = {'$':'USD','€':'EUR','£':'GBP'}.get(c, c)
            fields['total_amount'] = f"{amount} {currency}".strip() if currency else amount
            break

    return fields


def load_image(path):
    suffix = path.suffix.lower()
    if suffix == '.pdf':
        pages = convert_from_path(str(path), dpi=200, first_page=1, last_page=1)
        if not pages:
            raise ValueError(f"pdf2image returned no pages for {path.name}")
        return pages[0].convert('RGB')
    if suffix == '.avif':
        try:
            return Image.open(path).convert('RGB')
        except Exception:
            pass
        try:
            import tempfile, os
            with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as tmp:
                tmp_path = tmp.name
            subprocess.check_call(['convert', str(path), tmp_path], stderr=subprocess.DEVNULL)
            img = Image.open(tmp_path).convert('RGB')
            os.unlink(tmp_path)
            return img
        except (subprocess.CalledProcessError, FileNotFoundError):
            raise RuntimeError(
                f"Cannot open AVIF {path.name}.\n"
                "Install: pip install pillow-avif-plugin  OR  brew install imagemagick"
            )
    return Image.open(path).convert('RGB')


def ocr_image(image):
    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
    words, boxes = [], []
    w, h = image.size
    for i, text in enumerate(data['text']):
        text = str(text).strip()
        if not text or int(data['conf'][i]) < 10:
            continue
        left, top     = data['left'][i], data['top'][i]
        width, height = data['width'][i], data['height'][i]
        x0 = int(max(0, min(1000, round(left / w * 1000))))
        y0 = int(max(0, min(1000, round(top  / h * 1000))))
        x1 = int(max(0, min(1000, round((left + width)  / w * 1000))))
        y1 = int(max(0, min(1000, round((top  + height) / h * 1000))))
        if x1 <= x0: x1 = min(1000, x0 + 1)
        if y1 <= y0: y1 = min(1000, y0 + 1)
        words.append(text)
        boxes.append([x0, y0, x1, y1])
    return (words or ['empty']), (boxes or [[0, 0, 1, 1]])


# ── Main loop ──────────────────────────────────────────────────────────────
for img_path in MY_IMAGES:
    p = Path(img_path)
    print(f"\n{'='*65}")
    print(f"  {p.name}")
    print('='*65)

    if not p.exists():
        print(f"  File not found: {img_path}")
        continue

    try:
        if p.suffix.lower() == '.pdf' and _is_native_pdf(str(p)):
            # Native PDF — extract text directly, no OCR, no LayoutLMv3
            print(f"  Mode: native PDF text extraction")
            doc  = fitz.open(str(p))
            text = '\n'.join(page.get_text() for page in doc)
            doc.close()
            cleaned = _extract_fields_from_text(text)
            print(f"  {'FIELD':<20}  VALUE")
            print(f"  {'-'*20}  {'-'*35}")
            for field in FIELD_ORDER:
                val = cleaned.get(CLEAN_KEY[field], '') or '—'
                print(f"  {field:<20}  {val}")

        else:
            # Image or scanned PDF — OCR → LayoutLMv3 → InvoiceCleaner
            print(f"  Mode: OCR → LayoutLMv3 → InvoiceCleaner")
            image        = load_image(p)
            words, boxes = ocr_image(image)
            print(f"  OCR found {len(words)} words  |  size: {image.size}")
            raw     = get_raw_predictions(image, words, boxes)
            cleaned = cleaner.clean(raw, ocr_words=words)
            print(f"  {'FIELD':<20}  {'CLEANED':<28}  RAW")
            print(f"  {'-'*20}  {'-'*28}  {'-'*30}")
            for field in FIELD_ORDER:
                raw_val     = raw.get(field, '—')
                cleaned_val = cleaned.get(CLEAN_KEY[field], '') or '—'
                raw_display = (raw_val[:28] + '..') if len(raw_val) > 30 else raw_val
                print(f"  {field:<20}  {cleaned_val:<28}  {raw_display}")

    except Exception as e:
        print(f"  Error: {e}")
        raise



  doc_i_2.avif
  Mode: OCR → LayoutLMv3 → InvoiceCleaner
  OCR found 142 words  |  size: (802, 1133)
  FIELD                 CLEANED                       RAW
  --------------------  ----------------------------  ------------------------------
  INVOICE_NUMBER        12345                         Invoice no : 12345,
  INVOICE_DATE          —                             25 June
  DUE_DATE              —                             2022
  ISSUER_NAME           Ketut Susilo                  Ketut Susilo
  RECIPIENT_NAME        —                             —
  TOTAL_AMOUNT          —                             —

  doc_i_1.webp
  Mode: OCR → LayoutLMv3 → InvoiceCleaner
  OCR found 65 words  |  size: (1131, 1600)
  FIELD                 CLEANED                       RAW
  --------------------  ----------------------------  ------------------------------
  INVOICE_NUMBER        12345                         Invoice No. 12345
  INVOICE_DATE          —                             —
  DUE_DA

## 8. Token-level debug

Shows exactly which words the model tagged with which labels.
Use this when a field is missing or contains unexpected text.


In [8]:
# ── Config ────────────────────────────────────────────────────────────────
SPLIT             = 'test'
EXAMPLE_IDX       = 0
USE_CUSTOM_IMAGE  = False
CUSTOM_IMAGE_PATH = '/path/to/your/invoice.jpg'

# ── Load ───────────────────────────────────────────────────────────────────
if USE_CUSTOM_IMAGE:
    image        = Image.open(CUSTOM_IMAGE_PATH).convert('RGB')
    words, boxes = ocr_image(image)
    name         = Path(CUSTOM_IMAGE_PATH).name
else:
    example = raw_dataset[SPLIT][EXAMPLE_IDX]
    image   = Image.open(example['image_path']).convert('RGB')
    words   = example['words']
    boxes   = example['bboxes']
    name    = Path(example['image_path']).stem

print(f'Token predictions for: {name}')
print(f'Total OCR words      : {len(words)}')
print()
token_debug(image, words, boxes)


Token predictions for: Template1_Instance189
Total OCR words      : 98

LABEL                         WORD
-------------------------------------------------------
  B-RECIPIENT_NAME            : 'Bill'
  I-RECIPIENT_NAME            : 'to'
  I-RECIPIENT_NAME            : 'Shelly'
  I-RECIPIENT_NAME            : 'Rodriguez'
  I-RECIPIENT_NAME            : '02547'
  I-RECIPIENT_NAME            : 'Ramos'
  I-RECIPIENT_NAME            : 'Bypass'
  I-RECIPIENT_NAME            : 'Suite'
  I-RECIPIENT_NAME            : '849'
  I-RECIPIENT_NAME            : 'Williamshaven,'
  I-RECIPIENT_NAME            : 'NC'
  I-RECIPIENT_NAME            : '38767'
  I-RECIPIENT_NAME            : 'US'
  I-RECIPIENT_NAME            : 'Tel'
  I-RECIPIENT_NAME            : '+(463)893-0347'
  I-RECIPIENT_NAME            : 'Email'
  I-RECIPIENT_NAME            : 'christina14@example.org'
  I-RECIPIENT_NAME            : 'Site'
  I-RECIPIENT_NAME            : 'http'
  I-RECIPIENT_NAME            : '//www.gomez.com/'
